In [ ]:
# pip install "google-cloud-aiplatform>=1.66.0" "langchain-openai" "langchain-google-vertexai" "langchain>=0.2" pandas

### Environment and Vertex AI Initialization

Please replace `YOUR_GCP_PROJECT` with your Google Cloud Project ID and `YOUR_OPENAI_KEY` with your OpenAI API key.

In [ ]:
import os
import pandas as pd

from vertexai import init as vertex_init
from vertexai.evaluation import EvalTask
from vertexai.evaluation import types as eval_types

from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda


### 1. Environment and Vertex AI init

In [ ]:

PROJECT_ID = "YOUR_GCP_PROJECT" # Replace with your GCP project ID
LOCATION = "us-central1"  # region that supports Gen AI Eval
vertex_init(project=PROJECT_ID, location=LOCATION)

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"  # Replace with your OpenAI API key

### 2. Simple LangChain Agent Definition

This section defines the prompt and a helper function to create LangChain models.

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "You are a customer support agent. Answer concisely.\n\nQuestion: {question}"
)

def make_chain(model):
    return prompt | model

### OpenAI and Gemini Chain Initialization

Here, we initialize the `openai_chain` and `gemini_chain` using the `make_chain` function.

In [ ]:
openai_chain = make_chain(
    ChatOpenAI(
        model="gpt-4o",          # or another OpenAI model
        temperature=0.2,
    )
)

gemini_chain = make_chain(
    ChatVertexAI(
        model_name="gemini-2.0-flash-exp",  # or gemini-1.5-pro if preferred
        temperature=0.2,
    )
)

### Agent Runner Functions

These functions wrap the chains for easy reuse, particularly in evaluation tasks.

In [ ]:
def run_openai_agent(row):
    return openai_chain.invoke({"question": row["prompt"]}).content

def run_gemini_agent(row):
    return gemini_chain.invoke({"question": row["prompt"]}).content

### 3. Evaluation dataset (toy example)
In practice: load your customer-support eval set.

In [ ]:

eval_df = pd.DataFrame(
    [
        {
            "prompt": "How can I reset my account password?",
            "reference": "Explain using the Settings > Security > Reset password flow and email verification.",
        },
        {
            "prompt": "What is your refund policy for digital products?",
            "reference": "Describe the 30‑day refund window and exceptions for consumed credits.",
        },
    ]
)


### 4. Run both agents to create two “candidate” datasets

In [ ]:

def run_chain_over_df(df, agent_fn, candidate_name: str):
    rows = []
    for _, row in df.iterrows():
        output = agent_fn(row)
        rows.append(
            {
                "prompt": row["prompt"],
                "reference": row["reference"],
                "response": output,
                "candidate": candidate_name,
            }
        )
    return pd.DataFrame(rows)

openai_candidates = run_chain_over_df(eval_df, run_openai_agent, "openai_gpt4o")
gemini_candidates = run_chain_over_df(eval_df, run_gemini_agent, "gemini_flash")


### 5. Define an EvalTask using Gemini as LLM‑as‑a‑Judge
We use a pointwise text quality rubric + exact_match style metrics.

In [ ]:

metrics = [
    # Classic text metrics
    "exact_match",
    "rouge_l_sum",
    # LLM‑as‑a‑judge rubric metric (TEXT_QUALITY is a prebuilt rubric). [web:20][web:22][web:27]
    eval_types.RubricMetric.TEXT_QUALITY,
]

# Optional: custom rubric config (weights, prompt) can be added via RubricMetricConfig.

### 6. Evaluate OpenAI agent

In [ ]:

openai_task = EvalTask(
    dataset=openai_candidates,
    metrics=metrics,
    experiment="agent-eval-openai-vs-gemini",
    # judge_model left default → Gemini as judge in most configs. [web:23]
)

openai_result = openai_task.evaluate(
    # No model passed here; we already generated responses.
    experiment_run_name="openai-gpt4o-run",
)

print("=== OpenAI agent metrics ===")
openai_result.show()  # notebook-friendly visualization [web:20][web:22]

### 7. Evaluate Gemini agent

In [ ]:

gemini_task = EvalTask(
    dataset=gemini_candidates,
    metrics=metrics,
    experiment="agent-eval-openai-vs-gemini",
)

gemini_result = gemini_task.evaluate(
    experiment_run_name="gemini-flash-run",
)

print("=== Gemini agent metrics ===")
gemini_result.show()

### 8. Aggregate comparison (very simple)
#    You can also export to BigQuery or CSV for dashboards.

In [ ]:

openai_agg = openai_result.metrics_aggregate  # attribute name per SDK docs [web:22][web:27]
gemini_agg = gemini_result.metrics_aggregate

print("\n=== Side‑by‑side summary (text quality & exact_match) ===")
for metric_name in openai_agg:
    o_score = openai_agg[metric_name]["mean"]
    g_score = gemini_agg[metric_name]["mean"]
    print(f"{metric_name}: OpenAI={o_score:.3f} vs Gemini={g_score:.3f}")